# **Data Preparation**

This notebook was used to download the dataset, save it in Google Drive and prepare the data so that it has the correct format to use it in our RAG pipeline. It is important that the downloaded dataset has exactly this format (as we will create using this notebook) to execute the RAG pipeline.

Tommaso Giorgini

## Imports, Config and Utilities

In [ ]:
#imports
from google.colab import drive
from pathlib import Path
import tarfile
import re, os
import glob
import json
from tqdm import tqdm
!pip -q install datasets --upgrade
from datasets import Dataset, DatasetDict

drive.mount('/content/drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.4 MB/s eta 0:00:00
Mounted at /content/drive


## Download and Extraction of the Dataset


In [ ]:
# Destination Folder
BASE = "/content/drive/MyDrive/datasets/triviaqa"
!mkdir -p "$BASE"

!apt -yq install aria2

# Download
URL="https://nlp.cs.washington.edu/triviaqa/data/triviaqa-rc.tar.gz"
!aria2c -x 16 -s 16 -k 5M -d "$BASE" -o "triviaqa-rc.tar.gz" "$URL" || \
  wget -c "$URL" -O "$BASE/triviaqa-rc.tar.gz"


Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libaria2-0 libc-ares2
The following NEW packages will be installed:
  aria2 libaria2-0 libc-ares2
0 upgraded, 3 newly installed, 0 to remove and 41 not upgraded.
Need to get 1,513 kB of archives.
After this operation, 5,441 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libc-ares2 amd64 1.18.1-1ubuntu0.22.04.3 [45.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libaria2-0 amd64 1.36.0-1 [1,086 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 aria2 amd64 1.36.0-1 [381 kB]
Fetched 1,513 kB in 1s (1,377 kB/s)
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ..

In [ ]:
# .tar.gz original in drive
TAR = Path("/content/drive/MyDrive/datasets/triviaqa/triviaqa-rc.tar.gz")
assert TAR.exists(), f"Tar not founded: {TAR}"

# Were to save data
DATA_HOME_WIKI = Path("/content/drive/MyDrive/datasets/triviaqa_wiki")
DATA_HOME_WEB  = Path("/content/drive/MyDrive/datasets/triviaqa_web")
for base in [DATA_HOME_WIKI, DATA_HOME_WEB]:
    (base / "splits").mkdir(parents=True, exist_ok=True)
    (base / "hf" / "splits").mkdir(parents=True, exist_ok=True)
    (base / "hf" / "docstore").mkdir(parents=True, exist_ok=True)
    (base / "manifests").mkdir(parents=True, exist_ok=True)


# Selective Extraction (Wiki + Web) in /content
EXTRACT_BASE = Path("/content")

patterns = [
    # QA JSON - Wikipedia
    r'/qa/wikipedia-.*\.json$',
    r'/qa/verified-wikipedia-dev\.json$',
    r'/qa/wikipedia-dev\.json$',
    # QA JSON - Web
    r'/qa/web-.*\.json$',
    r'/qa/verified-web-dev\.json$',
    r'/qa/web-dev\.json$',
    # Evidence
    r'/evidence/wikipedia/.*',
    r'/evidence/web/.*',
]

def needed(member_name: str) -> bool:
    return any(re.search(p, "/" + member_name.strip("/")) for p in patterns)

with tarfile.open(TAR, 'r:gz') as tar:
    members = [m for m in tar.getmembers() if needed(m.name)]
    tar.extractall(path=str(EXTRACT_BASE), members=members)



/tmp/ipython-input-929348143.py:43: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=str(EXTRACT_BASE), members=members)


In [ ]:
# -------------------------
#UTILITIES

def load_items(p: Path):
    with p.open("r", encoding="utf-8") as f:
        obj = json.load(f)
    return obj.get("Data", obj)

def choose_dev_json(domain: str) -> Path:
    dev_cands = sorted(QA_DIR.glob(f"*{domain}*dev*.json"))
    verified = [p for p in dev_cands if "verified" in p.name.lower()]
    if verified:
        return verified[0]

    # fallback: largest
    def _count(p):
        try:
            return len(load_items(p))
        except:
            return -1
    return sorted(dev_cands, key=_count, reverse=True)[0]

def read_text(p: Path) -> str:
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc)
        except Exception:
            pass
    return p.read_bytes().decode(errors="ignore")

# ROOT
cands = glob.glob("/content/**/qa/wikipedia-train.json", recursive=True)
if cands:
    TRAIN_JSON_WIKI = Path(sorted(cands, key=len)[0])
    ROOT = TRAIN_JSON_WIKI.parents[1]
else:
    cands_web = glob.glob("/content/**/qa/web-train.json", recursive=True)
    TRAIN_JSON_WEB = Path(sorted(cands_web, key=len)[0])
    ROOT = TRAIN_JSON_WEB.parents[1]


QA_DIR   = ROOT / "qa"
WIKI_DIR = ROOT / "evidence" / "wikipedia"
WEB_DIR  = ROOT / "evidence" / "web"

print("ROOT:", ROOT)
print("QA_DIR:", QA_DIR)
print("WIKI_DIR exists:", WIKI_DIR.exists(), "| WEB_DIR exists:", WEB_DIR.exists())

ROOT: /content
QA_DIR: /content/qa
WIKI_DIR exists: True | WEB_DIR exists: True


## Dataset build in drive


In [ ]:
# DOMAIN BUILDING PIPELINE

def build_domain(data_home: Path, val_n: int):

    domain = "wikipedia"
    train_json = QA_DIR / "wikipedia-train.json"
    evidence_dir = WIKI_DIR
    page_key = "EntityPages"
    filename_field = "Filename"
    filenames_col = "Filenames"

    # Split
    dev_json = choose_dev_json(domain)
    print("TEST:", dev_json.name, "(", len(load_items(dev_json)), "examples )")

    train_all = load_items(train_json)
    val_items  = train_all[:val_n]
    train_items = train_all[val_n:]
    test_items  = load_items(dev_json)

    print(f"Split → train: {len(train_items):,} | val: {len(val_items):,} | test: {len(test_items):,}")

    # Picks only the information that we need from the original JSON file
    def to_flat(item):
        ans = item.get("Answer", {}) or {}
        aliases = ans.get("NormalizedAliases") or ans.get("Aliases") or []
        value   = ans.get("NormalizedValue") or ans.get("Value") or ""
        pages   = item.get(page_key, []) or []
        doc_paths = []
        for pg in pages:
            fn = pg.get(filename_field)
            if fn:
                doc_paths.append(str((evidence_dir / fn).resolve()))
        return {
            "QuestionId": item.get("QuestionId"),
            "Question": item.get("Question"),
            "Value": value,
            "Aliases": aliases,
            "doc_paths": doc_paths
        }

    # Write all stripped and cleaned JSON file for our processed dataset
    def write_jsonl(items, out_path: Path):
        miss = 0
        with out_path.open("w", encoding="utf-8") as w:
            for it in tqdm(items, desc=f"→ {domain}:{out_path.name}"):
                flat = to_flat(it)
                if flat["doc_paths"] and not any(Path(p).exists() for p in flat["doc_paths"]):
                    miss += 1
                w.write(json.dumps(flat, ensure_ascii=False) + "\n")

    SPLITS_DIR = data_home / "splits"
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    write_jsonl(train_items, SPLITS_DIR / "train.jsonl")
    write_jsonl(val_items,   SPLITS_DIR / "val.jsonl")
    write_jsonl(test_items,  SPLITS_DIR / "test.jsonl")


    # Building the HF dataset
    def to_rows(items):
        rows = []
        for it in items:
            ans = it.get("Answer", {}) or {}
            pages = it.get(page_key, []) or []
            rows.append({
                "QuestionId": it.get("QuestionId"),
                "Question": it.get("Question"),
                "Value": ans.get("NormalizedValue") or ans.get("Value") or "",
                "Aliases": (ans.get("NormalizedAliases") or ans.get("Aliases") or []),
                filenames_col: [p.get(filename_field) for p in pages if p.get(filename_field)],
            })
        return rows

    ds = DatasetDict({
        "train":      Dataset.from_list(to_rows(train_items)),
        "validation": Dataset.from_list(to_rows(val_items)),
        "test":       Dataset.from_list(to_rows(test_items)),
    })

    HF_SPLITS = data_home / "hf" / "splits"
    HF_SPLITS.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(HF_SPLITS))
    print(f"{domain}: HF splits salvati in:", HF_SPLITS)

    # Docstore all the evidence files, linked with a doc_idx to the question
    all_files = set()
    for split in ["train", "validation", "test"]:
        for rec in ds[split]:
            for fn in (rec.get(filenames_col) or []):
                all_files.add(fn)
    all_files = sorted(all_files)
    print(f"{domain}: Unique documents:", len(all_files))

    docs_rows, missing = [], 0
    for fn in tqdm(all_files, desc=f"{domain}:docstore"):
        abs_p = (evidence_dir / fn)
        if abs_p.exists():
            txt = read_text(abs_p)
            # normalization
            txt = re.sub(r'\s+', ' ', txt).strip()
            docs_rows.append({"doc_id": fn, "text": txt})
        else:
            missing += 1
    print(f"{domain}: Missing Documents:", missing)

    docs_ds = Dataset.from_list(docs_rows)
    HF_DOCS = data_home / "hf" / "docstore"
    HF_DOCS.mkdir(parents=True, exist_ok=True)
    docs_ds.save_to_disk(str(HF_DOCS))

    # Doc indexes
    doc_index = {row["doc_id"]: i for i, row in enumerate(docs_rows)}
    with (HF_DOCS / "doc_index.json").open("w", encoding="utf-8") as f:
        json.dump(doc_index, f)

Domain Build for Wikipedia dataset

In [ ]:
build_domain(domain="wikipedia", data_home=DATA_HOME_WIKI, val_n=7900)



=== COSTRUZIONE DOMINIO: wikipedia ===
Userò come TEST: verified-wikipedia-dev.json ( 318 esempi )
Split → train: 53,988 | val: 7,900 | test: 318


→ wikipedia:train.jsonl: 100%|██████████| 53988/53988 [00:07<00:00, 7441.77it/s]


wikipedia:train.jsonl: 53988 esempi | senza doc esistenti: 0


→ wikipedia:val.jsonl: 100%|██████████| 7900/7900 [00:01<00:00, 5453.55it/s]


wikipedia:val.jsonl: 7900 esempi | senza doc esistenti: 0


...

wikipedia: HF splits salvati in: /content/drive/MyDrive/datasets/triviaqa_wiki/hf/splits
wikipedia: Documenti unici referenziati: 42798


wikipedia:docstore: 100%|██████████| 42798/42798 [01:40<00:00, 423.76it/s]


wikipedia: Doc mancanti: 0


Saving the dataset (0/2 shards):   0%|          | 0/42798 [00:00<?, ? examples/s]

✅ wikipedia: Build completato. Manifest: /content/drive/MyDrive/datasets/triviaqa_wiki/manifests/manifest_wikipedia.json
